![KAUST Academy](https://i.imgur.com/a3uAqnb.png)

# Day 15 -- Lab 1: A Neural Network by Hand

Day 08 fitted a line to the Iris flowers with ten lines of gradient descent. Today you rebuild that neuron, measure its slopes with the chain rule, then add a hidden layer and run backpropagation through it, all in NumPy on 100 real flowers. Every gradient you compute is checked twice: against a tiny nudge of the dials (finite differences) and against PyTorch's autograd. By the end, `loss.backward()` is a line you can explain.

**Your role:** Fill in the `# Your code here` cells in order. Run every cell, including the ones already written for you.

**Dataset:** The Iris flowers (Fisher, 1936) shipped with scikit-learn: the 100 versicolor and virginica flowers, two columns (petal length, petal width), target 1 for virginica. The setup cell loads and scales them; no download needed.

---
# Setup

In [ ]:
# numpy, matplotlib, scikit-learn and torch are preinstalled on Colab. Uncomment if an import fails.
# !pip install -q numpy matplotlib scikit-learn torch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
np.set_printoptions(precision=4, suppress=True)

In [ ]:
iris = load_iris()
keep = iris.target > 0                                   # versicolor (1) and virginica (2) only
X_raw = iris.data[keep][:, 2:4]                           # petal length, petal width
y = (iris.target[keep] == 2).astype(float)                # 1 = virginica
mu, sd = X_raw.mean(axis=0), X_raw.std(axis=0)
X = (X_raw - mu) / sd                                     # scaled, as every model since Day 09
n = len(y)
print(X.shape, y.shape, "virginica share", y.mean())

---
# Part 1 -- One neuron

The *Definition: A Neuron Is Logistic Regression With a New Name* and *One Neuron, With Numbers* slides.

## Task 1: Look at the flowers

Draw a scatter plot of `X[:, 0]` against `X[:, 1]` with the colour given by `y` (`c=y, cmap="coolwarm"`), axis labels `petal length (scaled)` and `petal width (scaled)`, and a title. Can one straight line separate the two species almost perfectly?

In [ ]:
# Your code here


## Task 2: The sigmoid and the forward pass

Write `sigmoid(z)` and `forward(X, w, b)` that returns the neuron's output `a = sigmoid(X @ w + b)` for every row (shape `(100,)`). Test it with `w = np.array([1.0, -2.0])`, `b = -1.0`: `a[0]` should be about 0.514 and `a.mean()` about 0.330.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert a.shape == (100,) and abs(a[0] - 0.514) < 0.005, 'forward: sigmoid(X @ w + b), one number per row'
print('Task 2 passed')

## Task 3: The loss

Write `mse(a, y)` returning the mean squared error `np.mean((a - y) ** 2)` (the deck's loss; log loss works the same way, Day 09). Print it for the dials of Task 2: expected about 0.463.

In [ ]:
# Your code here


---
# Part 2 -- The gradient by the chain rule

The *Backprop Through One Neuron* slide: for every row, the error signal is `2 (a - y) a (1 - a)`; a dial's slope is the signal times the input the dial multiplies, averaged over rows.

## Task 4: Gradient of one neuron

Write `grad_neuron(X, y, w, b)` that returns `(grad_w, grad_b)`: compute `a = forward(X, w, b)`, the signal `delta = 2 * (a - y) * a * (1 - a)`, then `grad_w = X.T @ delta / n` (shape `(2,)`) and `grad_b = delta.mean()`. Print both for the Task 2 dials: expected `grad_w` about `[-0.166, -0.157]` and `grad_b` about 0.021.

In [ ]:
# Your code here


The check below nudges each dial by a tiny `eps` and measures how much the loss moves: the slope without any rule, as on the *Chain Rule, With Numbers* slide.

Run the check cell. It prints a message only when something needs another look.

In [ ]:
eps = 1e-5
num_w = np.array([(mse(forward(X, w + eps * e, b), y) - mse(forward(X, w - eps * e, b), y)) / (2 * eps) for e in np.eye(2)])
num_b = (mse(forward(X, w, b + eps), y) - mse(forward(X, w, b - eps), y)) / (2 * eps)
assert np.allclose(grad_w, num_w, atol=1e-6) and abs(grad_b - num_b) < 1e-6, 'grad_neuron: signal 2(a-y)a(1-a), then X.T @ delta / n and delta.mean()'
print('Task 4 passed: the chain rule agrees with the nudge test', num_w.round(4), round(num_b, 4))

## Task 5: Train the neuron

Starting from `w = np.zeros(2)`, `b = 0.0`, run 2,000 steps of the four moves with learning rate 1.0: gradient, then `w = w - lr * grad_w`, `b = b - lr * grad_b`. Store the loss of every step in a list `losses`, plot it, and print the final `w`, `b`, loss and accuracy (`np.mean((forward(X, w, b) > 0.5) == y)`). Expected: loss about 0.034, accuracy 0.94.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert losses[-1] < 0.06 and acc_neuron >= 0.9, 'train: 2,000 steps, lr 1.0, update w and b against their slopes'
print('Task 5 passed')

## Task 6: Compare with scikit-learn

Fit `LogisticRegression()` on `X, y` and print its accuracy on the same rows and its `coef_`. The dials differ (a different loss and a built-in penalty) but the accuracy should be the same 0.94: the neuron *is* logistic regression.

In [ ]:
# Your code here


---
# Part 3 -- A hidden layer

The *Forward Pass, With Numbers* and *Backprop Through a Hidden Layer* slides: three hidden ReLU neurons, then one sigmoid output.

In [ ]:
rng = np.random.default_rng(0)
W1, b1 = rng.normal(0, 1, size=(3, 2)), np.zeros(3)      # hidden layer: 3 neurons, 2 inputs each
w2, b2 = rng.normal(0, 1, size=3), 0.0                    # output neuron: reads the 3 hidden values
print("W1\n", W1, "\nw2", w2)

## Task 7: Forward pass with a hidden layer

Write `forward_net(X, W1, b1, w2, b2)` returning `(Z1, H, a)`: `Z1 = X @ W1.T + b1` (shape `(100, 3)`), `H = np.maximum(0, Z1)` (ReLU), `a = sigmoid(H @ w2 + b2)` (shape `(100,)`). Print the three shapes and `a[:3]`. Expected `a[:3]` about `[0.518, 0.480, 0.518]`.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert Z1.shape == (100, 3) and H.shape == (100, 3) and a.shape == (100,), 'forward_net: X @ W1.T + b1, ReLU, then H @ w2 + b2 through the sigmoid'
print('Task 7 passed')

## Task 8: Backprop through the hidden layer

Write `grad_net(X, y, W1, b1, w2, b2)` returning `(gW1, gb1, gw2, gb2)`. Steps from the slide: `d2 = 2 * (a - y) * a * (1 - a)` (signal at the output, shape `(100,)`); `gw2 = H.T @ d2 / n`, `gb2 = d2.mean()`; pass it back: `D1 = np.outer(d2, w2) * (Z1 > 0)` (shape `(100, 3)`: the signal times `w2`, switched off where ReLU was off); `gW1 = D1.T @ X / n`, `gb1 = D1.mean(axis=0)`. Print the four shapes.

In [ ]:
# Your code here


Two checks this time: the nudge test on one weight of `W1`, and PyTorch autograd on the whole network (the *Autograd Does It* slide).

Run the check cell. It prints a message only when something needs another look.

In [ ]:
def loss_net(W1, b1, w2, b2):
    return mse(forward_net(X, W1, b1, w2, b2)[2], y)
eps = 1e-5
bump = np.zeros_like(W1); bump[1, 0] = eps
num = (loss_net(W1 + bump, b1, w2, b2) - loss_net(W1 - bump, b1, w2, b2)) / (2 * eps)
assert abs(gW1[1, 0] - num) < 1e-6, 'grad_net: check D1 = outer(d2, w2) * (Z1 > 0) and gW1 = D1.T @ X / n'
tW1, tb1 = torch.tensor(W1, requires_grad=True), torch.tensor(b1, requires_grad=True)
tw2, tb2 = torch.tensor(w2, requires_grad=True), torch.tensor(b2, requires_grad=True)
ta = torch.sigmoid(torch.relu(torch.tensor(X) @ tW1.T + tb1) @ tw2 + tb2)
((ta - torch.tensor(y)) ** 2).mean().backward()
assert np.allclose(tW1.grad.numpy(), gW1, atol=1e-6) and np.allclose(tw2.grad.numpy(), gw2, atol=1e-6), 'grad_net disagrees with autograd'
print('Task 8 passed: nudge test and autograd both agree with your backprop')

## Task 9: Train the network

Train the 2-3-1 network for 3,000 steps with learning rate 0.5 (same four moves, now for four dials), recording the loss. Plot the loss, then print the final loss and accuracy (`a > 0.5`). Expected: loss about 0.034 and accuracy 0.94, the same as the single neuron: on these two columns a line was already the best boundary.

In [ ]:
# Your code here


Run the check cell. It prints a message only when something needs another look.

In [ ]:
assert losses_net[-1] < 0.05 and acc_net >= 0.93, 'train the network: 3,000 steps, lr 0.5, every dial moves against its own slope'
print('Task 9 passed')

## Task 10: What the network draws

Draw the decision regions of the trained neuron and of the trained network side by side: build a grid with `np.meshgrid(np.linspace(-2.5, 2.5, 200), np.linspace(-2.5, 2.5, 200))`, predict on the grid points, and `contourf` the 0.5 cut with the flowers on top. Does the network's boundary bend, or did the hidden layer settle on a line?

In [ ]:
# Your code here


---
## Task 11: Reflect

The nudge test and autograd agreed with your chain-rule gradients every time: in one sentence, what does `loss.backward()` compute, and why is `requires_grad=True` needed first? The network tied the neuron at 94 of 100 flowers here. Why no gain, and on which kind of data would the gap be large (think of the Tahoe cars and the MNIST digits of the deck)? Your hidden layer had 3 neurons. What would change with 30, and what would you watch to decide (Day 09)?

*Your answers here*

---
## Conclusion

- **A neuron** is Day 08's line plus a bend: you wrote its forward pass and got the same accuracy as scikit-learn's logistic regression.
- **The chain rule** gave the neuron's slopes, and a tiny nudge of each dial confirmed them without any rule.
- **Backpropagation** passed the output's error signal back through the hidden layer; the nudge test and PyTorch autograd agreed with your numbers.
- **Training** was the same four moves for every dial at once, and the network tied the neuron on data where a line already fits.

---

Made By **Sattam Altwaim**